# Challenge Lab 4 — Workflow Agent Pipeline

Multi-perspective research pipeline using all three ADK workflow agents:
- **ParallelAgent** — two researchers search simultaneously from different angles
- **SequentialAgent** — chains research → synthesis → review
- **LoopAgent** — iterative critique/refine cycle with early exit

In [1]:
!pip install google-adk

## Configure Vertex AI

In [2]:
import os

project_id = !gcloud config get-value project
PROJECT_ID = project_id[0].strip()
print(f"Project ID: {PROJECT_ID}")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

print("Vertex AI backend configured")

Project ID: qwiklabs-gcp-00-93a45da1459f
Vertex AI backend configured


## Project structure

In [3]:
!mkdir -p research_pipeline

In [4]:
%%writefile research_pipeline/__init__.py


Overwriting research_pipeline/__init__.py


In [5]:
%%bash
cat > research_pipeline/.env << EOF
GOOGLE_CLOUD_PROJECT=$(gcloud config get-value project)
GOOGLE_CLOUD_LOCATION=global
GOOGLE_GENAI_USE_VERTEXAI=TRUE
EOF
cat research_pipeline/.env

GOOGLE_CLOUD_PROJECT=qwiklabs-gcp-00-93a45da1459f
GOOGLE_CLOUD_LOCATION=global
GOOGLE_GENAI_USE_VERTEXAI=TRUE


## Agent definitions

```
greeter (LlmAgent)
  └── answer_pipeline (SequentialAgent)
        ├── research_team (ParallelAgent)
        │     ├── primary_researcher → output_key="primary_research"
        │     └── counter_researcher → output_key="counter_research"
        ├── synthesizer → output_key="draft_answer"
        └── review_loop (LoopAgent, max_iterations=2)
              ├── critic → output_key="critique"
              └── refiner + exit_loop → output_key="draft_answer"
```

In [6]:
%%writefile research_pipeline/agent.py
from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.tools import google_search
from google.adk.tools.tool_context import ToolContext
from google.adk.models.google_llm import Gemini
from google.genai import types

MODEL = Gemini(
    model="gemini-2.5-flash",
    retry_options=types.HttpRetryOptions(initial_delay=1, attempts=3),
)


# ── exit tool ────────────────────────────────────────────────────────────────

def exit_loop(tool_context: ToolContext):
    """Signal that the review loop should stop — answer is good enough."""
    tool_context.actions.escalate = True
    return {"status": "loop_exited"}


# ── Parallel researchers ─────────────────────────────────────────────────────

primary_researcher = LlmAgent(
    name="primary_researcher",
    model=MODEL,
    description="Searches for direct answers and supporting evidence.",
    instruction=(
        "Search for factual information to answer the user's question. "
        "Use multiple searches if needed. Cite sources.\n\n"
        "IMPORTANT: Keep your response to bullet points and key data. "
        "Do NOT write an essay or conclusion. Just report what you found."
    ),
    tools=[google_search],
    output_key="primary_research",
)

counter_researcher = LlmAgent(
    name="counter_researcher",
    model=MODEL,
    description="Devil's advocate — finds problems, gaps, and opposing evidence.",
    instruction=(
        "You are a devil's advocate. Your ONLY job is to find criticisms, "
        "counterarguments, and things the obvious answer gets wrong.\n\n"
        "Search specifically for: skeptics, dissenting experts, failed predictions, "
        "hidden costs, industry bias, and inconvenient data.\n\n"
        "Do NOT write a balanced analysis — someone else does that. "
        "ONLY report the problems and pushback. Keep it to bullet points."
    ),
    tools=[google_search],
    output_key="counter_research",
)

research_team = ParallelAgent(
    name="research_team",
    sub_agents=[primary_researcher, counter_researcher],
)


# ── Synthesizer ──────────────────────────────────────────────────────────────

synthesizer = LlmAgent(
    name="synthesizer",
    model=MODEL,
    description="Combines both research streams into a single balanced draft.",
    include_contents="none",
    instruction=(
        "You have two research inputs — one mainstream, one adversarial. "
        "Combine them into a single clear answer that addresses the user's question.\n\n"
        "MAIN FINDINGS:\n{primary_research}\n\n"
        "COUNTERPOINTS:\n{counter_research}\n\n"
        "Rules:\n"
        "- Be direct. Don't hedge on settled facts.\n"
        "- Where the counterpoints raise legitimate issues, address them honestly.\n"
        "- Where the counterpoints are weak or industry-funded, say so.\n"
        "- Keep it under 400 words."
    ),
    output_key="draft_answer",
)


# ── Review loop: critic + refiner ────────────────────────────────────────────

critic = LlmAgent(
    name="critic",
    model=MODEL,
    description="Fact-checks the draft and flags issues.",
    include_contents="none",
    instruction=(
        "Review this draft answer:\n\n"
        "DRAFT: {draft_answer}\n\n"
        "Check for:\n"
        "- Factual errors or unsupported claims\n"
        "- Missing important context the user asked about\n"
        "- One-sided framing or bias\n"
        "- Vague hedging where a clear answer exists\n\n"
        "If the answer is solid, respond with exactly: APPROVED\n"
        "Otherwise, list the specific issues. Be concrete."
    ),
    output_key="critique",
)

refiner = LlmAgent(
    name="refiner",
    model=MODEL,
    description="Improves the draft based on critique, or exits loop if approved.",
    include_contents="none",
    instruction=(
        "DRAFT: {draft_answer}\n\n"
        "CRITIQUE: {critique}\n\n"
        "If the critique says APPROVED, call the exit_loop tool immediately. "
        "Do not output any text.\n\n"
        "Otherwise, rewrite the draft to fix every issue the critic raised. "
        "Keep it under 400 words. Output only the improved answer."
    ),
    tools=[exit_loop],
    output_key="draft_answer",
)

review_loop = LoopAgent(
    name="review_loop",
    description="Iterates critique and refinement until the answer passes.",
    sub_agents=[critic, refiner],
    max_iterations=2,
)


# ── Full pipeline ────────────────────────────────────────────────────────────

answer_pipeline = SequentialAgent(
    name="answer_pipeline",
    description="Multi-perspective research, synthesis, then iterative review.",
    sub_agents=[research_team, synthesizer, review_loop],
)


# ── Root agent ───────────────────────────────────────────────────────────────

root_agent = LlmAgent(
    name="greeter",
    model=MODEL,
    description="Entry point — routes questions to the research pipeline.",
    instruction=(
        "You are a research assistant. When the user asks a question, "
        "immediately delegate to answer_pipeline. Keep your greeting to one sentence."
    ),
    sub_agents=[answer_pipeline],
)

Overwriting research_pipeline/agent.py


## Verify files

In [7]:
!ls -la research_pipeline/

total 32
drwxr-xr-x 4 root root 4096 Mar  5 20:21 .
drwxr-xr-x 7 root root 4096 Mar  5 20:11 ..
drwxr-xr-x 3 root root 4096 Mar  5 20:24 .adk
-rw-r--r-- 1 root root 5865 Mar  5 20:25 agent.py
-rw-r--r-- 1 root root  110 Mar  5 20:25 .env
-rw-r--r-- 1 root root    1 Mar  5 20:25 __init__.py
drwxr-xr-x 2 root root 4096 Mar  5 20:21 __pycache__


## Test with InMemoryRunner

In [8]:
import importlib.util
from google.adk.runners import InMemoryRunner
from google.genai import types

# Load agent module
spec = importlib.util.spec_from_file_location("research_pipeline", "research_pipeline/agent.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

runner = InMemoryRunner(agent=mod.root_agent, app_name="research_pipeline")
print("Runner ready")


async def run_query(query, user_id="test_user"):
    """Send a query and print all agent events to show the workflow."""
    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print(f"{'='*70}")

    session = await runner.session_service.create_session(
        app_name="research_pipeline",
        user_id=user_id,
    )

    message = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=message,
    ):
        author = event.author or "system"

        # Show agent transfers
        if hasattr(event, "actions") and event.actions:
            if hasattr(event.actions, "transfer_to_agent") and event.actions.transfer_to_agent:
                print(f"  [{author}] >>> transfer to: {event.actions.transfer_to_agent}")
            if hasattr(event.actions, "escalate") and event.actions.escalate:
                print(f"  [{author}] >>> escalate (exit loop)")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    fc = part.function_call
                    args = dict(fc.args) if fc.args else {}
                    # Truncate long args for readability
                    args_str = str(args)[:120]
                    print(f"  [{author}] tool call: {fc.name}({args_str})")
                elif hasattr(part, "function_response") and part.function_response:
                    fr = part.function_response
                    resp_str = str(dict(fr.response) if fr.response else {})[:150]
                    print(f"  [{author}] tool result: {fr.name} -> {resp_str}")
                elif hasattr(part, "text") and part.text:
                    text = part.text.strip()
                    if text:
                        # Show first 300 chars of text responses
                        preview = text[:300] + ("..." if len(text) > 300 else "")
                        print(f"  [{author}]: {preview}")

    print()

Runner ready


### Test 1 — debatable topic (parallel researchers should diverge)

In [9]:
await run_query("Is nuclear energy a good solution for climate change?")


QUERY: Is nuclear energy a good solution for climate change?


  [greeter] tool call: transfer_to_agent({'agent_name': 'answer_pipeline'})
  [greeter] >>> transfer to: answer_pipeline
  [greeter] tool result: transfer_to_agent -> {'result': None}
  [counter_researcher]: Here are some criticisms and counterarguments regarding nuclear energy as a solution for climate change:

*   **Unresolved Waste Management Issues:** No country has successfully disposed of high-level nuclear waste or spent nuclear fuel permanently. This waste remains radioactive for hundreds of tho...
  [primary_researcher]: Nuclear energy presents a complex picture as a solution for climate change, offering significant benefits in terms of emissions reduction but also posing substantial challenges.

Key data points include:

*   **Low Carbon Emissions:**
    *   Nuclear power plants produce no greenhouse gas emissions ...
  [synthesizer]: Nuclear energy presents a dual nature in the fight against climate change. It offers significant advantages by producing virtually no greenhous

### Test 2 — factual question (counter_researcher should confirm consensus)

In [10]:
await run_query("What caused the 2024 CrowdStrike outage?")


QUERY: What caused the 2024 CrowdStrike outage?
  [greeter] tool call: transfer_to_agent({'agent_name': 'answer_pipeline'})
  [greeter] >>> transfer to: answer_pipeline
  [greeter] tool result: transfer_to_agent -> {'result': None}
  [primary_researcher]: The 2024 CrowdStrike outage, which occurred on July 19, 2024, was caused by a faulty configuration update to CrowdStrike's Falcon Sensor security software.

Key details include:
*   **Cause:** A "logic error" in a configuration update (specifically, Channel File 291) for the Falcon sensor software f...
  [counter_researcher]: Here are criticisms, counterarguments, and problems related to the 2024 CrowdStrike outage:

*   **Single Point of Failure Risk:** The widespread reliance on CrowdStrike's Falcon platform created a single point of failure, amplifying the impact of the faulty update across numerous organizations glob...
  [synthesizer]: The 2024 CrowdStrike outage on July 19 was caused by a "logic error" in a configuration update

### Test 3 — opinion-heavy (should benefit most from the review loop)

In [11]:
await run_query("Should I learn Rust or Go for backend development in 2026?")


QUERY: Should I learn Rust or Go for backend development in 2026?
  [greeter] tool call: transfer_to_agent({'agent_name': 'answer_pipeline'})
  [greeter] >>> transfer to: answer_pipeline
  [greeter] tool result: transfer_to_agent -> {'result': None}
  [primary_researcher]: Here's an overview of Rust and Go for backend development in 2026:

**Rust for Backend Development in 2026:**

*   **Performance & Efficiency:**
    *   Offers performance comparable to C/C++ due to its compilation to highly optimized machine code and zero-cost abstractions.
    *   Consistently dem...
  [counter_researcher]: Here are some criticisms and counterarguments regarding Rust and Go for backend development in 2026:

**Rust Criticisms and Challenges:**

*   **Steep Learning Curve and Productivity Hit:**
    *   Rust's concepts like ownership, borrowing, and lifetimes present a notoriously steep learning curve, m...
  [synthesizer]: For backend development in 2026, both Rust and Go are strong contenders, eac

## Run via CLI (optional)

In [12]:
# Interactive CLI — Ctrl+C to stop
!adk run research_pipeline

Log setup complete: /tmp/agents_log/agent.20260305_203110.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
/usr/local/lib/python3.12/dist-packages/google/adk/cli/cli.py:189: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
Running agent greeter, type exit to exit.
[user]: Compare the total environmental impact of electric vehicles versus gas cars, including manufacturing, battery mining,    grid source, and end-of-life disposal.
[counter_researcher]: Here ar

In [ ]:
# Web UI — Ctrl+C to stop
!adk web